# Track C (DNA): Continuous Objective vs Categorical CE on Real Genomic DNA

**The Evo 2 analog of synthetic Variant A.** Hybrid `SmallStripedHyena` backbone
(Hyena long-convolution + attention stripes), real human chr22 windowed to 512 bp,
identical optimizer / steps / lr / schedule / precision / seed (320). The only
thing that changes between conditions is the output objective.

Same logic as the protein track, with two differences:

- **Backbone:** `SmallStripedHyena` (autoregressive / causal), mirroring Evo 2.
  Objective is next-token (CLM); recovery accuracy = next-token recovery.
- **Perturbation suite adds reverse-complement**, the symmetry test that ties this
  result directly back to the Evo 2 texture finding (a DNA-track-only probe the
  protein track cannot run).

## Conditions (all share backbone, data, schedule, seed=320)

| Condition | Head | Loss | Decode |
|---|---|---|---|
| **CE** | `Linear(d_model, vocab)` | cross-entropy over A/C/G/T + specials | argmax |
| **Cont-physchem** | `Linear(d_model, d_code)` | MSE to structural code (purine / amino / strong-H-bond / mass) | nearest-prototype |
| **Cont-random** | `Linear(d_model, d_code)` | MSE to fixed random orthonormal code | nearest-prototype |

The DNA structural code is **non-monotone in token index** and identity-preserving
(it uniquely separates the four bases), so it avoids both failure modes: the
staircase leak (a monotone index rescaling) and going inert (collapsing the bases
onto too few scalars). `Cont-random` is again the confound control.

## Setup
1. Upload `utils/evaluation_harness.py`, `utils/perturbation_protocol.py`, and
   `utils/bio_continuous_codes.py` (or keep this notebook under `geometric-alignment-tax/`).
2. GPU runtime. The representation-forming layers must be trained from scratch
   under each objective (a frozen-backbone head won't work).

---

In [ ]:
# Install dependencies (Colab)
print("Installing dependencies...")
!pip install -q torch transformers datasets shesha-geometry matplotlib seaborn pandas scipy scikit-learn

import os, sys, gc, time, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, '.')
sys.path.insert(0, './utils')
sys.path.insert(0, '../utils')
sys.path.insert(0, '../../utils')
sys.path.insert(0, '../../../utils')

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration (seed=320 everywhere)

SEED = 320
PHASE = 'full'   # 'quick' for a fast smoke run, 'full' for the headline result

CONFIG = {
    'quick': dict(n_train=4_000,  n_eval=1_000, epochs=4, batch_size=64),
    'full':  dict(n_train=50_000, n_eval=2_000, epochs=8, batch_size=64),
}[PHASE]

# --- data / tokenization ---
SEQ_LEN = 512                      # bp per window (matched to synthetic setup)

# --- backbone (SmallStripedHyena; matched across all conditions) ---
D_MODEL  = 256
N_LAYERS = 4
N_HEADS  = 4
ORDER    = 2
MLP_RATIO = 4
DROPOUT  = 0.1

# --- continuous code ---
D_CODE   = 4                       # structural code dimensionality

# --- objective / optimization (identical across conditions) ---
LR           = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS       = CONFIG['epochs']
BATCH_SIZE   = CONFIG['batch_size']
N_TRAIN      = CONFIG['n_train']
N_EVAL       = CONFIG['n_eval']

# --- perturbation suite (adds reverse-complement) ---
SUB_RATES = [0.01, 0.02, 0.05, 0.10]

# --- Shesha harness ---
N_BOOTSTRAP = 5 if PHASE == 'full' else 0
MAX_SAMPLES = 2500

# --- paths (Google Drive on Colab, local fallback otherwise) ---
SAVE_TO_DRIVE = True                         # mount Drive and persist everything there
DRIVE_SUBDIR  = 'track_c_dna_continuous'     # folder under MyDrive/geometric-alignment-tax/
LOCAL_BASE    = './results/track_c_dna_continuous/'

def _resolve_output_base():
    """Mount Google Drive (Colab) and return a base dir under MyDrive.
    Falls back to a local path off Colab, if mount fails, or if SAVE_TO_DRIVE is off.
    RESULTS_DIR / CACHE_DIR / CKPT_DIR are derived from this, so flipping the
    base is the only change needed to move all outputs to Drive."""
    if not SAVE_TO_DRIVE:
        return LOCAL_BASE
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        base = f'/content/drive/MyDrive/geometric-alignment-tax/{DRIVE_SUBDIR}/'
        print(f"Saving outputs to Google Drive: {base}")
        return base
    except Exception as e:
        print(f"Google Drive unavailable ({e}); saving locally to {LOCAL_BASE}")
        return LOCAL_BASE

OUTPUT_BASE = _resolve_output_base()
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'
for d in (RESULTS_DIR, CACHE_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print(f"OUTPUT_BASE = {OUTPUT_BASE}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=SEED):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

set_seed(SEED)
print(f"Phase: {PHASE.upper()} | device: {DEVICE}")
print(f"Train/eval windows: {N_TRAIN}/{N_EVAL} | seq_len: {SEQ_LEN} bp | "
      f"epochs: {EPOCHS} | d_model: {D_MODEL} | d_code: {D_CODE}")

In [ ]:
# Fixed continuous codes + shared tokenization (DNA)
from bio_continuous_codes import (
    build_dna_codes, nearest_prototype_decode, recovery_accuracy,
    procrustes_distortion, rdm_similarity as np_rdm_similarity, NUCLEOTIDES,
)

CODEBOOKS = {
    'physchem':  build_dna_codes(kind='physchem', d=D_CODE,    seed=SEED),
    'random':    build_dna_codes(kind='random',   d=D_CODE,    seed=SEED),
    # Output-dimensionality control: d == VOCAB_SIZE (same head width as CE).
    # 4 nucleotides in 9-d space -> fully orthonormal (K_REAL=4 <= VOCAB_SIZE=9).
    'random_hd': build_dna_codes(kind='random',   d=VOCAB_SIZE, seed=SEED),
}

TOK2ID     = CODEBOOKS['physchem'].token_to_id
VOCAB_SIZE = CODEBOOKS['physchem'].vocab_size
K_REAL     = len(NUCLEOTIDES)                  # decodable tokens 0..3 == A,C,G,T
MASK_ID = TOK2ID['<mask>']; PAD_ID = TOK2ID['<pad>']; UNK_ID = TOK2ID['<unk>']

assert CODEBOOKS['physchem'].token_to_id == CODEBOOKS['random'].token_to_id
# A<->T, C<->G complement is exactly (K_REAL-1 - id) under the 'ACGT' ordering.
assert NUCLEOTIDES == ['A', 'C', 'G', 'T']

def encode_dna(seq, seq_len=SEQ_LEN):
    ids = [TOK2ID.get(c, UNK_ID) for c in seq[:seq_len]]
    ids = ids + [PAD_ID] * max(0, seq_len - len(ids))
    return np.asarray(ids[:seq_len], dtype=np.int64)

print(f"VOCAB_SIZE={VOCAB_SIZE} | decodable nucleotides K={K_REAL} | D_CODE={D_CODE}")
for kind, cb in CODEBOOKS.items():
    orth = 'orthonormal' if K_REAL <= cb.d else 'normalized Gaussian'
    print(f"  {kind:10s}: codes {cb.codes.shape} | min_pairwise={cb.min_pairwise_distance():.4f} "
          f"| {orth} | feats={cb.feature_names}")
print("\nIdentity is recoverable from exact codes:")
for kind, cb in CODEBOOKS.items():
    dec = nearest_prototype_decode(cb.prototypes, cb)
    print(f"  {kind:10s}: exact-code nearest-prototype accuracy = {(dec==cb.prototype_ids).mean()*100:.1f}%")

In [ ]:
# Load real genomic DNA (human chr22), windowed to SEQ_LEN bp
# Reuses the project's chr22 loader (UCSC hg38). Synthetic fallback if offline.

import urllib.request, gzip

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')


DNA_PROVENANCE = None


def load_chr22_windows(n_sequences, seq_length=SEQ_LEN, seed=SEED, chrom_url=CHR22_URL, tag='chr22'):
    global DNA_PROVENANCE
    cache = f"{CACHE_DIR}/{tag}_{n_sequences}_{seq_length}_{seed}.txt"
    if os.path.exists(cache):
        with open(cache) as f:
            seqs = [ln.strip() for ln in f if ln.strip()]
        DNA_PROVENANCE = 'cached'
        print(f"Loaded {len(seqs)} cached {tag} windows (DNA_PROVENANCE=cached)")
        return seqs
    rng = np.random.default_rng(seed)
    _real_loaded = False
    try:
        print(f"Downloading {tag} (~50MB, first run only)...")
        with urllib.request.urlopen(chrom_url, timeout=120) as resp:
            with gzip.GzipFile(fileobj=resp) as fh:
                lines = fh.read().decode('utf-8').split('\n')
        genome = ''.join(ln.strip() for ln in lines[1:] if ln.strip()).upper()
        print(f"  {tag}: {len(genome):,} bp")
        seqs = []
        for _ in range(int(n_sequences * 1.5)):
            start = int(rng.integers(0, len(genome) - seq_length))
            win = genome[start:start + seq_length]
            if sum(c not in VALID_BASES for c in win) < seq_length * 0.10:
                seqs.append(''.join(c if c in VALID_BASES else rng.choice(list('ACGT')) for c in win))
            if len(seqs) >= n_sequences:
                break
        _real_loaded = True
        DNA_PROVENANCE = 'real:ucsc_hg38'
    except Exception as e:
        print("\n" + "!" * 70)
        print(f"WARNING: chr22 download FAILED ({e}). USING SYNTHETIC FALLBACK.")
        print("Synthetic random DNA has no conserved structure. Results from this")
        print("run do NOT support any biological geometry claim.")
        print("!" * 70 + "\n")
        seqs = [''.join(rng.choice(list('ACGT'), size=seq_length)) for _ in range(n_sequences)]
        DNA_PROVENANCE = 'SYNTHETIC_FALLBACK'
    seqs = seqs[:n_sequences]
    with open(cache, 'w') as f:
        f.write('\n'.join(seqs))
    print(f"Cached {len(seqs)} windows to {cache}")
    return seqs


_all = load_chr22_windows(N_TRAIN + N_EVAL, seed=SEED)
rng = np.random.default_rng(SEED); rng.shuffle(_all)
train_seqs, eval_seqs = _all[:N_TRAIN], _all[N_TRAIN:N_TRAIN + N_EVAL]
train_ids = np.stack([encode_dna(s) for s in train_seqs])
eval_ids  = np.stack([encode_dna(s) for s in eval_seqs])
print(f"\ntrain_ids {train_ids.shape} | eval_ids {eval_ids.shape}")
for b, base in enumerate('ACGT'):
    print(f"  {base}: {(train_ids == b).mean()*100:.1f}%")
print(f"\nDNA_PROVENANCE: {DNA_PROVENANCE}")
assert DNA_PROVENANCE != 'SYNTHETIC_FALLBACK', (
    "Real genomic data failed to load. Delete the cache and fix the network "
    "connection before running the experiment. Synthetic DNA produces "
    "meaningless geometry results."
)

In [ ]:
# Perturbation suite (SNP + reverse-complement) + dual-head SmallStripedHyena
# Backbone mirrors the project's SmallStripedHyena (Evo 2 analog); only the head
# (and hence loss) changes between conditions.

def perturb_substitute(ids, rate, rng):
    out = ids.copy()
    for i in range(out.shape[0]):
        pos = np.where(out[i] < K_REAL)[0]
        if len(pos) == 0:
            continue
        for p in rng.choice(pos, size=max(1, int(len(pos) * rate)), replace=False):
            alt = int(rng.integers(0, K_REAL))
            while alt == out[i, p]:
                alt = int(rng.integers(0, K_REAL))
            out[i, p] = alt
    return out


def reverse_complement_ids(ids):
    """A<->T, C<->G then reverse: complement(id) = (K_REAL-1) - id for 'ACGT'."""
    out = ids.copy()
    real = out < K_REAL
    comp = out.copy()
    comp[real] = (K_REAL - 1) - out[real]
    return comp[:, ::-1].copy()


def build_perturbations(ids, seed=SEED):
    rng = np.random.default_rng(seed)
    pert = {f"snp_{int(r*100)}pct": perturb_substitute(ids, r, rng) for r in SUB_RATES}
    pert['reverse_complement'] = reverse_complement_ids(ids)
    return pert


class ImplicitFilterMLP(nn.Module):
    """Hyena implicit long-convolution filter."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        n_pos = 16
        self.pos_emb = nn.Linear(n_pos, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._pos(seq_len, n_pos))
        self.seq_len = seq_len

    def _pos(self, seq_len, n):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        pf = self.pos_features if seq_len == self.seq_len else self._pos(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, short_conv_kernel, padding=short_conv_kernel // 2, groups=d_model)
            for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]; n = 2 * L
        return torch.fft.irfft(torch.fft.rfft(signal, n=n, dim=-1) *
                               torch.fft.rfft(kernel, n=n, dim=-1).unsqueeze(0), n=n, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Causal multi-head attention with RoPE (StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _rope(self, x):
        L = x.shape[2]; f = self.freqs[:L]
        cos_f, sin_f = torch.cos(f)[None, None], torch.sin(f)[None, None]
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._rope(q), self._rope(k)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask[None, None], float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        h = int(d_model * mlp_ratio)
        self.mlp_gate, self.mlp_value, self.mlp_out = nn.Linear(d_model, h), nn.Linear(d_model, h), nn.Linear(h, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        g = self.norm2(x)
        return x + self.mlp_out(F.silu(self.mlp_gate(g)) * self.mlp_value(g))


class SmallStripedHyena_Bio(nn.Module):
    """SmallStripedHyena (Evo 2 analog), dual head.

    objective='ce'   -> Linear(d_model, vocab_size), cross-entropy.
    objective='cont' -> Linear(d_model, d_code),     MSE to the fixed code.
    """
    def __init__(self, objective='ce', d_code=D_CODE, vocab_size=VOCAB_SIZE,
                 d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, seq_len=SEQ_LEN,
                 order=ORDER, mlp_ratio=MLP_RATIO, attention_layers=None, dropout=DROPOUT):
        super().__init__()
        assert objective in ('ce', 'cont')
        self.objective = objective
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))   # ~1 attention stripe / 4 layers
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order,
                              is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size if objective == 'ce' else d_code, bias=False)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.drop(self.tok_emb(x))
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        out = self.head(h)
        return (out, h) if return_hidden else out


for obj in ('ce', 'cont'):
    _m = SmallStripedHyena_Bio(obj)
    n_attn = len(_m.attention_layers)
    print(f"SmallStripedHyena_Bio ({obj}): {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params "
          f"({N_LAYERS-n_attn} Hyena + {n_attn} Attn)")
    del _m
print("Perturbation suite + dual-head SmallStripedHyena ready")

In [ ]:
# CLM (next-token) training + recovery + embeddings + geometry snapshots
from evaluation_harness import StabilityHarness

harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30,
                           seed=SEED, max_samples=MAX_SAMPLES, n_bootstrap=N_BOOTSTRAP)


@torch.no_grad()
def eval_recovery(model, objective, codebook, ids, batch_size=128):
    """Next-token recovery accuracy over real positions (argmax vs nearest-prototype)."""
    model.eval(); correct = total = 0
    for i in range(0, len(ids), batch_size):
        chunk = ids[i:i + batch_size]
        x = torch.from_numpy(chunk).long().to(DEVICE)
        out = model(x[:, :-1]).cpu().numpy()
        tgt = chunk[:, 1:]
        real = tgt < K_REAL
        pred = out[..., :K_REAL].argmax(-1) if objective == 'ce' else nearest_prototype_decode(out, codebook)
        correct += int((pred[real] == tgt[real]).sum()); total += int(real.sum())
    return correct / max(total, 1)


@torch.no_grad()
def extract_embeddings(model, ids, batch_size=128):
    model.eval(); outs = []
    for i in range(0, len(ids), batch_size):
        x = torch.from_numpy(ids[i:i + batch_size]).long().to(DEVICE)
        _, h = model(x, return_hidden=True)
        mask = (x < K_REAL).unsqueeze(-1).float()
        outs.append(((h * mask).sum(1) / mask.sum(1).clamp(min=1.0)).cpu().numpy())
    return np.concatenate(outs, 0)


SNAP_N = min(512, N_EVAL)
SNAP_CLEAN = eval_ids[:SNAP_N]
SNAP_PERT = perturb_substitute(SNAP_CLEAN, 0.05, np.random.default_rng(SEED))


def geometry_snapshot(model, objective, codebook):
    rec = eval_recovery(model, objective, codebook, SNAP_CLEAN)
    Xc = extract_embeddings(model, SNAP_CLEAN); Xp = extract_embeddings(model, SNAP_PERT)
    return dict(recovery=rec,
                procrustes_D=procrustes_distortion(Xc, Xp),
                rdm_sim=np_rdm_similarity(Xc, Xp))


def train_condition(objective, codebook=None, epochs=EPOCHS, log=True):
    """Train one condition. Head width is inferred from the codebook so
    Cont-highdim (d == VOCAB_SIZE) shares the same output width as CE."""
    set_seed(SEED)
    d_out = codebook.d if (objective == 'cont' and codebook is not None) else VOCAB_SIZE
    model = SmallStripedHyena_Bio(objective, d_code=d_out).to(DEVICE)
    code_tensor = torch.from_numpy(codebook.codes).float().to(DEVICE) if objective == 'cont' else None
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_batches = (N_TRAIN + BATCH_SIZE - 1) // BATCH_SIZE
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * n_batches)
    Xtrain = torch.from_numpy(train_ids).long()
    order = np.arange(N_TRAIN); history = []
    for ep in range(epochs):
        model.train(); np.random.default_rng(SEED + ep).shuffle(order)
        ep_loss = nb = 0
        for s in range(0, N_TRAIN, BATCH_SIZE):
            batch = Xtrain[order[s:s + BATCH_SIZE]].to(DEVICE)
            inp, tgt = batch[:, :-1], batch[:, 1:]
            out = model(inp)
            real = tgt < K_REAL
            if objective == 'ce':
                loss = F.cross_entropy(out[real], tgt[real])
            else:
                loss = F.mse_loss(out[real], code_tensor[tgt[real]])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            ep_loss += loss.item(); nb += 1
        snap = geometry_snapshot(model, objective, codebook)
        snap.update(epoch=ep + 1, train_loss=ep_loss / max(nb, 1)); history.append(snap)
        if log:
            print(f"    ep {ep+1:2d}/{epochs} loss={snap['train_loss']:.4f} "
                  f"recovery={snap['recovery']*100:5.1f}%  D={snap['procrustes_D']:.4f}  "
                  f"rdm_sim={snap['rdm_sim']:.4f}")
    return model, history


print("Training / recovery / embedding / snapshot helpers ready (CLM)")

In [ ]:
# Run all conditions: train -> recovery -> geometry (Shesha + Procrustes D + reverse-complement)
import pandas as pd

CONDITIONS = [
    ('CE',             'ce',   None),
    ('Cont-physchem',  'cont', 'physchem'),
    ('Cont-random',    'cont', 'random'),
    # Output-dimensionality control: d == VOCAB_SIZE; same head width as CE.
    # If this arm also shows lower D, the geometry gain is not from low-d alone.
    ('Cont-highdim',   'cont', 'random_hd'),
]

perturbed_ids = build_perturbations(eval_ids, seed=SEED)
trained, histories, rows, clean_emb = {}, {}, [], {}

for name, obj, cbkey in CONDITIONS:
    print('=' * 70); print(f"CONDITION: {name}"); print('=' * 70)
    cb = CODEBOOKS[cbkey] if cbkey else None
    model, hist = train_condition(obj, cb)
    trained[name], histories[name] = model, hist

    rec = eval_recovery(model, obj, cb, eval_ids)
    Xc = extract_embeddings(model, eval_ids); clean_emb[name] = Xc
    print(f"  full-eval next-token recovery: {rec*100:.2f}%")

    for pname, pids in perturbed_ids.items():
        Xp = extract_embeddings(model, pids)
        res = harness.evaluate(model_name=name, embeddings_clean=Xc,
                               embeddings_perturbed=Xp, perturbation_name=pname)
        D = procrustes_distortion(Xc, Xp)
        rows.append(dict(condition=name, objective=obj, code=cbkey or 'none',
                         perturbation=pname, recovery_acc=rec, procrustes_D=D,
                         rdm_similarity=res.rdm_similarity_score,
                         composite_stability=res.composite_stability,
                         pert_stability=res.perturbation_stability_score))
        print(f"    {pname:18s}  D={D:.4f}  rdm_sim={res.rdm_similarity_score:.4f}  "
              f"composite={res.composite_stability:.4f}")
    cleanup_gpu()

df = pd.DataFrame(rows)
df.to_csv(f"{RESULTS_DIR}/dna_geometry_detailed.csv", index=False)

agg = (df.groupby(['condition', 'code'])
         .agg(recovery_acc=('recovery_acc', 'first'),
              mean_procrustes_D=('procrustes_D', 'mean'),
              mean_rdm_similarity=('rdm_similarity', 'mean'),
              mean_composite=('composite_stability', 'mean'))
         .reset_index())
agg.to_csv(f"{RESULTS_DIR}/dna_geometry_summary.csv", index=False)
print('\n' + '=' * 70); print("GEOMETRY SUMMARY (lower D = more stable; higher rdm_sim = better)")
print('=' * 70)
print(agg.to_string(index=False))

# Reverse-complement is the DNA-specific symmetry test (Evo 2 texture link).
rc = df[df['perturbation'] == 'reverse_complement']
print("\nReverse-complement distortion D (symmetry test):")
for _, r in rc.iterrows():
    print(f"  {r['condition']:14s}: D={r['procrustes_D']:.4f}  rdm_sim={r['rdm_similarity']:.4f}")

In [ ]:
# Frozen linear probe on a real downstream task
# Real task = a Nucleotide-Transformer benchmark (promoter / enhancer / splice-site).
# Probe is logistic regression on FROZEN embeddings. Bar: Cont retains comparable
# accuracy to CE while showing the geometry improvement.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score


def load_downstream_dna(candidates=('promoter_all', 'enhancers', 'splice_sites_all'),
                        max_train=4000, max_test=1500):
    """Returns (Xtr, ytr, Xte, yte, task_name).

    Raises RuntimeError if *all* candidate tasks fail, so a silent skip never
    hides a broken probe from the results table.
    """
    from datasets import load_dataset  # ImportError propagates loud if missing
    last_err = None
    for cfg in candidates:
        try:
            ds = load_dataset('InstaDeepAI/nucleotide_transformer_downstream_tasks', cfg)
            tr = ds['train']; te = ds['test'] if 'test' in ds else ds['validation']

            # Verify label column before iterating the full split.
            sample = next(iter(tr))
            if 'label' not in sample:
                raise RuntimeError(
                    f"'label' column missing in NT_downstream[{cfg}]. "
                    f"Available columns: {list(sample.keys())}"
                )

            def grab(split, m):
                seqs, labs = [], []
                for r in split:
                    s = (r.get('sequence') or '').upper()
                    s = ''.join(c for c in s if c in 'ACGT')
                    if len(s) >= 30:
                        seqs.append(s); labs.append(int(r['label']))
                    if len(seqs) >= m:
                        break
                return seqs, np.array(labs)

            Xtr, ytr = grab(tr, max_train); Xte, yte = grab(te, max_test)
            if len(ytr) == 0:
                raise RuntimeError(f"No sequences collected from NT_downstream[{cfg}]")
            keep = np.isin(yte, np.unique(ytr))
            Xte = [s for s, k in zip(Xte, keep) if k]; yte = yte[keep]
            print(f"  Probe data: task={cfg} train={len(ytr)} test={len(yte)} "
                  f"classes={len(np.unique(ytr))}")
            return Xtr, ytr, Xte, yte, f"NT_downstream[{cfg}]"
        except Exception as e:
            last_err = e; continue
    raise RuntimeError(
        f"All NT downstream tasks failed to load. Last error: {last_err}. "
        f"Check HuggingFace access and dataset availability before running."
    )


def probe_condition(name, Xtr_seq, ytr, Xte_seq, yte):
    model = trained[name]
    Etr = extract_embeddings(model, np.stack([encode_dna(s) for s in Xtr_seq]))
    Ete = extract_embeddings(model, np.stack([encode_dna(s) for s in Xte_seq]))
    sc = StandardScaler().fit(Etr)
    clf = LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(Etr), ytr)
    pred = clf.predict(sc.transform(Ete))
    return balanced_accuracy_score(yte, pred), accuracy_score(yte, pred)


probe_rows = []
Xtr_seq, ytr, Xte_seq, yte, task_name = load_downstream_dna()
print(f"Downstream task: {task_name}")
for name, _, _ in CONDITIONS:
    bal, acc = probe_condition(name, Xtr_seq, ytr, Xte_seq, yte)
    probe_rows.append(dict(condition=name, probe_balanced_acc=bal, probe_acc=acc))
    print(f"  {name:14s}  balanced_acc={bal*100:5.2f}%  acc={acc*100:5.2f}%")
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(f"{RESULTS_DIR}/dna_probe.csv", index=False)
print('\n' + probe_df.to_string(index=False))
print("\nPrimary task metric (probe reads frozen representations). Bar: Cont arms "
      "within a few points of CE = comparable performance + better geometry.")

In [ ]:
# Matched-performance check: geometry gap persists at equal next-token recovery.
# Focus on Cont-random and Cont-highdim for the primary comparison: these arms
# have no biological ceiling on recovery, so there is full overlap with CE's range.
import matplotlib.pyplot as plt

mh = pd.DataFrame([dict(condition=name, **h) for name, hist in histories.items() for h in hist])
mh.to_csv(f"{RESULTS_DIR}/dna_matched_history.csv", index=False)


def interp_D_at(hist, levels):
    h = sorted(hist, key=lambda d: d['recovery'])
    recs = np.array([d['recovery'] for d in h]); Ds = np.array([d['procrustes_D'] for d in h])
    return np.interp(levels, recs, Ds)


ce_hist = histories['CE']; ce_recs = [d['recovery'] for d in ce_hist]
print("Procrustes D at MATCHED recovery accuracy (CE vs Cont):\n")
matched_rows = []
for cont in ['Cont-physchem', 'Cont-random', 'Cont-highdim']:
    ch = histories[cont]; c_recs = [d['recovery'] for d in ch]
    lo, hi = max(min(ce_recs), min(c_recs)), min(max(ce_recs), max(c_recs))
    if hi <= lo:
        print(f"  {cont}: no recovery-accuracy overlap with CE (cannot match)."); continue
    levels = np.linspace(lo, hi, 5)
    Dce, Dcont = interp_D_at(ce_hist, levels), interp_D_at(ch, levels)
    print(f"  {cont} vs CE  (overlap recovery {lo*100:.1f}-{hi*100:.1f}%):")
    for lv, dce, dco in zip(levels, Dce, Dcont):
        print(f"    recovery={lv*100:5.1f}%   D_CE={dce:.4f}   D_{cont}={dco:.4f}   gap={dce-dco:+.4f}")
        matched_rows.append(dict(cont=cont, recovery=lv, D_CE=dce, D_cont=dco, gap=dce - dco))
if matched_rows:
    pd.DataFrame(matched_rows).to_csv(f"{RESULTS_DIR}/dna_matched_D.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
colors = {'CE': '#DC2626', 'Cont-physchem': '#2563EB', 'Cont-random': '#16A34A'}
for name, hist in histories.items():
    recs = [d['recovery'] * 100 for d in hist]; Ds = [d['procrustes_D'] for d in hist]
    ax.plot(recs, Ds, 'o-', color=colors.get(name), label=name, lw=2, ms=6)
ax.set_xlabel('Next-token recovery accuracy (%)'); ax.set_ylabel('Procrustes distortion D (5% SNP)')
ax.set_title('Matched-performance check: geometry gap persists at equal recovery', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/dna_matched_performance.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Headline figure: comparable accuracy + better geometry (+ reverse-complement)
import matplotlib.pyplot as plt

colors = {
    'CE':            '#DC2626',
    'Cont-physchem': '#2563EB',
    'Cont-random':   '#16A34A',
    'Cont-highdim':  '#7C3AED',
}
names = list(colors)
snp_df = df[df['perturbation'].str.startswith('snp')].copy()
snp_df['rate'] = snp_df['perturbation'].str.extract(r'(\d+)').astype(int)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

ax = axes[0]
for n in names:
    sub = snp_df[snp_df['condition'] == n].sort_values('rate')
    ax.plot(sub['rate'], sub['procrustes_D'], 'o-', color=colors[n], label=n, lw=2)
ax.set_xlabel('SNP rate (%)'); ax.set_ylabel('Procrustes distortion D')
ax.set_title('A. Geometric distortion (lower = better)', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for n in names:
    sub = snp_df[snp_df['condition'] == n].sort_values('rate')
    ax.plot(sub['rate'], sub['rdm_similarity'], 'o-', color=colors[n], label=n, lw=2)
ax.set_xlabel('SNP rate (%)'); ax.set_ylabel('RDM similarity')
ax.set_title('B. Relational geometry (higher = better)', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
x = np.arange(len(names)); w = 0.38
rec = [df[df['condition'] == n]['recovery_acc'].iloc[0] * 100 for n in names]
if len(probe_df):
    prb = [float(probe_df[probe_df['condition'] == n]['probe_balanced_acc'].iloc[0]) * 100
           if (probe_df['condition'] == n).any() else np.nan for n in names]
else:
    prb = [np.nan] * len(names)
ax.bar(x - w/2, rec, w, label='next-token recovery', color=[colors[n] for n in names], alpha=0.9)
ax.bar(x + w/2, prb, w, label='probe balanced acc', color=[colors[n] for n in names], alpha=0.5, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15); ax.set_ylabel('accuracy (%)')
ax.set_title('C. Task performance (must be comparable)', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3, axis='y')

ax = axes[3]
rc_D = [df[(df['condition'] == n) & (df['perturbation'] == 'reverse_complement')]['procrustes_D'].iloc[0] for n in names]
ax.bar(x, rc_D, color=[colors[n] for n in names])
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15); ax.set_ylabel('Procrustes D under RC')
ax.set_title('D. Reverse-complement symmetry (Evo 2 link)', fontweight='bold'); ax.grid(alpha=0.3, axis='y')

fig.suptitle('Track C (DNA): continuous objective reduces geometric distortion '
             'while retaining task performance', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/dna_headline.png", dpi=200, bbox_inches='tight')
plt.show()

ce_D = df[df['condition'] == 'CE']['procrustes_D'].mean()
print('\nVERDICT (mean Procrustes D across perturbations):')
for n in names:
    d = df[df['condition'] == n]['procrustes_D'].mean()
    print(f"  {n:14s}: D={d:.4f}" + (f"  ({ce_D/d:.2f}x lower distortion than CE)" if n != 'CE' and d > 0 else ""))

## What this buys

On real genomic DNA, under matched conditions, this track supplies the DNA half
of the bridge:

> A continuous regression objective (MSE to a fixed per-nucleotide code) reduces
> geometric distortion **D** by a measured factor relative to categorical
> cross-entropy while retaining comparable next-token recovery and frozen-probe
> downstream accuracy.

### Controlled variable callout (for the writeup)

This track uses **causal CLM** (`SmallStripedHyena`), matching Evo 2's native
training objective. The synthetic Variant A also uses CLM, so there is no
uncontrolled deviation here. Within this track, the *only* thing that varies
is CE vs MSE-to-code. State this explicitly so readers can verify the comparison
is single-variable.

### Three confounds closed

| Confound | Control |
|---|---|
| Biological content in target | `Cont-random` — arbitrary target, same objective form |
| Optimization scale / loss landscape | Matched-performance check (geometry gap at equal recovery) |
| **Output dimensionality** | `Cont-highdim` — d == VOCAB\_SIZE, same head width as CE |

### Reverse-complement framing

The reverse-complement perturbation is a **related symmetry probe** — it tests
whether the representation respects the Watson-Crick complement symmetry under
mean-pooling. This is *not* the same quantity as the Evo 2 per-sequence
RC-consistency result (which measures token-level consistency). Frame it as
a complementary symmetry probe rather than a replication of that finding.

### Orthonormality note

`Cont-random` (d=4, K=4) is **fully orthonormal** (K ≤ d). `Cont-highdim`
(d=VOCAB\_SIZE=9, K=4) is also **fully orthonormal**. Both are biologically
arbitrary; neither code contains conserved genomic information.

### Optional strengtheners
- Add a **held-out chromosome** (chr21) for a held-out-genome geometry test.
- Sweep `D_MODEL` / `N_LAYERS` to show the CE geometry gap does not close with scale.
- Augment the DNA code with dinucleotide physicochemical features (raise `D_CODE`).

### Outputs (under `results/track_c_dna_continuous/results/`)
- `dna_geometry_detailed.csv`, `dna_geometry_summary.csv`
- `dna_probe.csv`
- `dna_matched_history.csv`, `dna_matched_D.csv`
- `dna_headline.png`, `dna_matched_performance.png`

Together with `Track_C_Protein_Continuous.ipynb`, this converts the synthetic
Variant-A causal proof into a causal claim about real biological sequence models.